# Customer Churn Prediction - Data Preparation

This notebook prepares the dataset for model training. The main things to handle are `TotalCharges`, the target column, categorical features and the train/test split.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

data_path = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)
df.head()


## Clean the columns

`TotalCharges` is read as text because a few rows contain blank values. I convert it to numeric and let the preprocessing pipeline handle the missing values.


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges:", df["TotalCharges"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())


## Separate features and target


In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Features:", X.shape)
print("Target counts:")
print(y.value_counts())


## Train and test sets

The split is stratified so the churn ratio stays similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training churn rate:", round(y_train.mean(), 3))
print("Test churn rate:", round(y_test.mean(), 3))


## Build the preprocessing pipeline


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


## Check the prepared data


In [ ]:
print("Missing values after preprocessing:", np.isnan(X_train_processed).sum())
print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)
